<a href="https://colab.research.google.com/github/yenlung/Python-Math-AI/blob/main/10%E7%A5%9E%E7%A7%98%E7%9A%84%E7%86%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10 神秘的熵 🎲
## 用數學測量「驚喜感」和「混亂程度」

這週我們要學的是資訊理論裡最重要的概念——**熵 (Entropy)**。

> 你有沒有想過：**「告訴你一件事，到底傳了多少資訊？」**

---

### 先想想這兩件事：

**場景 A**：你住在一個幾乎每天都是晴天的地方。
- 有人說：「明天是晴天。」→ 你的反應：「廢話！」（資訊量 ≈ 0）
- 有人說：「明天是颱風！」→ 你的反應：「什麼！？」（資訊量超大！）

**場景 B**：老師說「下週要考試」，全班都已經知道了——這句話完全沒有資訊量。

---

### 這週要學什麼？

| 概念 | 白話意思 | AI 應用 |
|------|---------|--------|
| **資訊量** | 一件事有多「驚喜」 | 壓縮編碼 |
| **熵 (Entropy)** | 平均驚喜感 = 「亂度」 | 決策樹分裂依據 |
| **Cross Entropy** | 用「預測」來衡量「現實」的誤差 | AI 訓練的損失函數 |
| **KL 散度** | 兩個分佈差多遠 | 生成式 AI 的核心工具 |

這些概念是現代 AI（包括 ChatGPT！）訓練的基礎，學完你就能理解 AI 是怎麼「學習」的。

## 🤖 可以這樣問 AI

這週隨時可以問 AI：

- 「資訊量為什麼要用 -log(P) 定義？可以用別的嗎？」
- 「熵和物理學中的熵有什麼關係？」
- 「cross entropy 和 MSE（均方誤差）什麼時候用哪個？」
- 「KL 散度對稱嗎？DKL(P||Q) 和 DKL(Q||P) 一樣嗎？」
- 「為什麼 log 要用 2 為底？用 e 不行嗎？」
- 「決策樹怎麼用熵來決定要用哪個特徵分裂？」

**記住：先自己想一下，再問 AI，印象才會深。**

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

## 0. 先搞定中文顯示

一樣，先把 Google 的開源中文字型請進來。

In [ ]:
!wget -q https://github.com/notofonts/noto-cjk/raw/refs/heads/main/Sans/OTF/TraditionalChinese/NotoSansCJKtc-Medium.otf
print('字型下載完成 ✅')

In [ ]:
import matplotlib.font_manager as fm

fm.fontManager.addfont('NotoSansCJKtc-Medium.otf')
mpl.rc('font', family='Noto Sans CJK TC')
mpl.rcParams['axes.unicode_minus'] = False

print('中文字型設定完成 ✅')

# 1. 資訊量：用數字測量「驚喜感」📢

## 1-1. 定義資訊量

我們想定義一個函數 $I(P)$，表示「機率為 $P$ 的事件發生時，帶來多少資訊量」。

這個函數應該滿足：

- 機率越**小**，資訊量越**大**（颱風比晴天驚喜）
- 機率 = 1（必然發生），資訊量 = 0（廢話）
- 機率 = 0（不可能發生），資訊量 = 無限大

數學家給出的答案：

$$I(P) = -\log_2(P)$$

用 $\log_2$ 是因為這樣**資訊量的單位是「位元 (bit)」**，和電腦編碼直接對應！

> 以 8 種等機率事件為例：$P = 1/8$，資訊量 $= -\log_2(1/8) = 3$ bits。
> 剛好就是「用 3 個 bit 可以編碼 8 種可能」！

In [ ]:
# 定義資訊量函數
def information(p):
    """計算機率 p 的事件所帶來的資訊量 (bits)"""
    return -np.log2(p)

# 幾個例子
examples = {
    '丟公平硬幣出現正面 (P=1/2)': 1/2,
    '丟公平骰子出現 6 (P=1/6)': 1/6,
    '8種等機率事件之一 (P=1/8)': 1/8,
    '晴天城市預報晴天 (P=0.9)': 0.9,
    '晴天城市預報颱風 (P=0.02)': 0.02,
}

print('資訊量計算範例：')
print('-' * 50)
for desc, p in examples.items():
    print(f'{desc}')
    print(f'  → 資訊量 = {information(p):.3f} bits')
    print()

In [ ]:
# 畫出資訊量函數的圖形
p = np.linspace(0.001, 1, 500)
I = information(p)

plt.figure(figsize=(8, 5))
plt.plot(p, I, color='steelblue', linewidth=2.5)
plt.xlabel('事件發生的機率 P')
plt.ylabel('資訊量 I(P)（bits）')
plt.title('資訊量函數：$I(P) = -\log_2(P)$')

# 標記幾個特殊點
for p_val, label in [(0.5, '1/2'), (1/6, '1/6'), (1/8, '1/8'), (0.9, '0.9')]:
    i_val = information(p_val)
    plt.plot(p_val, i_val, 'ro', markersize=7)
    plt.annotate(f'P={label}\nI={i_val:.1f}', xy=(p_val, i_val),
                 xytext=(p_val + 0.05, i_val + 0.3), fontsize=9,
                 arrowprops=dict(arrowstyle='->', color='gray'))

plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.xlim(0, 1)
plt.ylim(-0.5, 11)
plt.tight_layout()
plt.show()

print('機率越小 → 資訊量越大，機率 = 1 → 資訊量 = 0，完全符合直覺！')

## 1-2. 資訊量和編碼的關係

資訊量不只是數學遊戲，它和「**最佳編碼需要幾個 bit**」直接相關！

| 事件數 | 機率（均等） | 資訊量 | 需要幾個 bit 編碼 |
|--------|------------|--------|------------------|
| 2 種 | 1/2 | 1 bit | 1 bit（0, 1）|
| 4 種 | 1/4 | 2 bits | 2 bits（00, 01, 10, 11）|
| 8 種 | 1/8 | 3 bits | 3 bits（000~111）|
| 26 個英文字母 | 1/26 | ≈ 4.7 bits | 5 bits（足夠）|

這告訴我們：**把常見事件放前面（短編碼），罕見事件放後面（長編碼）** → 這就是「霍夫曼編碼」的核心想法！

In [ ]:
# 驗證一下 26 個英文字母
n_events = 26
p_each = 1 / n_events
info_bits = information(p_each)
print(f'26 個英文字母（假設等機率）：')
print(f'  每個字母的機率 = 1/26 ≈ {p_each:.4f}')
print(f'  資訊量 = {info_bits:.3f} bits')
print(f'  所以用 5 個 bit 編碼（2^5=32 種）就夠了！')

# 2. 熵 (Entropy)：平均資訊量 🌀

## 2-1. 熵的定義

如果一個機率分布有 $n$ 個事件，機率分別是 $p_1, p_2, \ldots, p_n$，
則**平均資訊量**（也就是資訊量的期望值）叫做**熵**：

$$H(P) = -\sum_{i=1}^{n} p_i \log_2(p_i)$$

直覺：

- 所有事件機率**相等**（最亂）→ 熵**最大**
- 某個事件機率接近 1（幾乎確定）→ 熵**接近 0**
- 熵可以理解成「**亂度**」或「**不確定性**」

In [ ]:
def entropy(probs):
    """計算機率分布的熵（bits）
    probs: list 或 numpy array，所有元素加起來要等於 1
    """
    probs = np.array(probs)
    # 避免 log(0) 的問題：機率為 0 的項對熵的貢獻是 0
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

# 幾種不同的天氣分布
distributions = {
    '永遠是晴天 [1, 0, 0]': [1, 0, 0],
    '晴雨各半 [0.5, 0.5, 0]': [0.5, 0.5, 0],
    '三種天氣均等 [1/3, 1/3, 1/3]': [1/3, 1/3, 1/3],
    '晴天常見 [0.7, 0.2, 0.1]': [0.7, 0.2, 0.1],
    '骰子六面等機率': [1/6]*6,
    '硬幣兩面等機率': [0.5, 0.5],
}

print('各種機率分布的熵：')
print('-' * 50)
for name, dist in distributions.items():
    h = entropy(dist)
    print(f'{name}')
    print(f'  → H = {h:.4f} bits')
    print()

## 2-2. 熵的視覺化：亂度的感覺

我們來比較幾種分布，感受一下熵代表的「亂度」。

In [ ]:
# 用二元分布（只有兩種事件）的熵，看 p 從 0 到 1 的變化
p_vals = np.linspace(0.001, 0.999, 500)
H_binary = np.array([entropy([p, 1-p]) for p in p_vals])

plt.figure(figsize=(8, 5))
plt.plot(p_vals, H_binary, color='darkorange', linewidth=2.5)
plt.xlabel('事件 A 的機率 p（事件 B 的機率 = 1-p）')
plt.ylabel('熵 H（bits）')
plt.title('二元分布的熵（當 p=0.5，熵最大 = 1 bit）')

# 標記最大值
plt.plot(0.5, 1.0, 'ro', markersize=10, label='最大值 H=1 (p=0.5，最亂)')
plt.plot(0.01, entropy([0.01, 0.99]), 'bs', markersize=8, label='H≈0 (幾乎確定，最不亂)')
plt.plot(0.99, entropy([0.99, 0.01]), 'bs', markersize=8)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 視覺化幾種不同「亂度」的分布
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

cases = [
    {'probs': [0.98, 0.01, 0.01],
     'labels': ['晴天', '雨天', '颱風'],
     'title': '幾乎確定（晴天城市）'},
    {'probs': [0.7, 0.2, 0.1],
     'labels': ['晴天', '雨天', '颱風'],
     'title': '有點不確定'},
    {'probs': [1/3, 1/3, 1/3],
     'labels': ['晴天', '雨天', '颱風'],
     'title': '完全不知道（最亂）'},
]

colors = ['#FFD700', '#87CEEB', '#708090']

for ax, case in zip(axes, cases):
    h = entropy(case['probs'])
    bars = ax.bar(case['labels'], case['probs'], color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f"{case['title']}\n熵 H = {h:.3f} bits", fontsize=11)
    ax.set_ylabel('機率')
    ax.set_ylim(0, 1.1)
    for bar, p in zip(bars, case['probs']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{p:.2f}', ha='center', fontsize=10)

plt.suptitle('同樣三種天氣，不同的機率分布 → 不同的熵', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('越「均勻」的分布，熵越大！越「確定」的分布，熵越小。')

## 2-3. 熵和編碼：最佳壓縮

熵告訴我們：**最佳編碼平均需要幾個 bit**。

如果機率分布非常「不均勻」，代表某些事件很常發生——我們就應該給它短一點的編碼，省空間！

In [ ]:
# 假設一個語言只有 4 個字母，但機率不同
# 字母:  A     B     C     D
probs_lang = [0.5,  0.25, 0.125, 0.125]
letters = ['A', 'B', 'C', 'D']

h = entropy(probs_lang)
print(f'這個語言的熵 = {h:.3f} bits')
print()
print('最佳霍夫曼編碼（常見的用短編碼）：')
codes = {'A': '0', 'B': '10', 'C': '110', 'D': '111'}
avg_bits = sum(probs_lang[i] * len(codes[letters[i]]) for i in range(4))
print(f'{"字母":<6} {"機率":<8} {"編碼":<8} {"長度"}')
print('-' * 35)
for letter, prob in zip(letters, probs_lang):
    code = codes[letter]
    print(f'{letter:<6} {prob:<8.3f} {code:<8} {len(code)} bits')
print('-' * 35)
print(f'平均長度 = {avg_bits:.3f} bits')
print(f'最佳熵   = {h:.3f} bits')
print(f'→ 霍夫曼編碼幾乎達到理論最佳！')

# 3. Cross Entropy：AI 的誤差函數 🤖

## 3-1. 從辨識問題說起

假設我們要做一個**人臉辨識系統**，區分三個人：**小潔、小純、小佑**。

- 輸入：一張照片
- 輸出：三個數字（每個人的機率）→ 加起來等於 1

假設這張照片是**小潔**：
- **正確答案** $P = [1, 0, 0]$（小潔 100%，其他 0%）
- **AI 的預測** $Q = [0.70, 0.19, 0.11]$（AI 說有 70% 機率是小潔）

**這個誤差要怎麼量？**

---

## 3-2. Cross Entropy 的定義

如果正確答案是分布 $P = [p_1, p_2, \ldots, p_n]$，AI 預測是分布 $Q = [q_1, q_2, \ldots, q_n]$，
則 **Cross Entropy** 定義為：

$$H(P, Q) = -\sum_{i=1}^{n} p_i \log_2(q_i)$$

**直覺**：
- 用正確答案 $P$ 來「加權」，但取 $Q$（AI 預測）的資訊量
- $Q$ 越接近 $P$，cross entropy 越小（誤差越小）
- 完全一樣時，cross entropy = 熵 $H(P)$，這是理論最低值

在 AI 訓練中，**cross entropy 就是損失函數**——AI 要最小化它！

In [ ]:
def cross_entropy(P, Q):
    """計算 P 和 Q 的 cross entropy
    P: 正確答案的機率分布
    Q: 預測的機率分布
    """
    P = np.array(P, dtype=float)
    Q = np.array(Q, dtype=float)
    # 只看 P > 0 的項（P=0 的項貢獻是 0）
    mask = P > 0
    return -np.sum(P[mask] * np.log2(Q[mask]))

# 人臉辨識例子
# 正確答案：小潔 (index 0)
P_correct = [1, 0, 0]  # 正確答案

predictions = {
    '完美預測 [1.0, 0.0, 0.0]': [1.0, 0.0, 0.0],
    '不錯的預測 [0.9, 0.07, 0.03]': [0.9, 0.07, 0.03],
    '尚可的預測 [0.7, 0.19, 0.11]': [0.7, 0.19, 0.11],
    '很差的預測 [0.3, 0.4, 0.3]': [0.3, 0.4, 0.3],
    '完全錯誤 [0.0, 1.0, 0.0]': [0.001, 0.998, 0.001],  # 避免 log(0)
}

print(f'正確答案 P = {P_correct} (小潔 = 100%)')
print()
print(f'{"預測 Q":<35} {"Cross Entropy"}')
print('-' * 55)
for name, Q in predictions.items():
    ce = cross_entropy(P_correct, Q)
    bar = '█' * int(ce * 5)
    print(f'{name:<35} {ce:.4f}  {bar}')

In [ ]:
# 視覺化：當正確答案是 [1,0,0]，預測的第一個機率 q1 從 0 到 1 變化時的 cross entropy
q1_vals = np.linspace(0.001, 0.999, 500)
ce_vals = [cross_entropy([1, 0, 0], [q1, (1-q1)/2, (1-q1)/2]) for q1 in q1_vals]

plt.figure(figsize=(8, 5))
plt.plot(q1_vals, ce_vals, color='crimson', linewidth=2.5)
plt.xlabel('AI 預測「小潔」的機率 q₁')
plt.ylabel('Cross Entropy（誤差）')
plt.title('正確答案是小潔時，AI 的誤差（Cross Entropy）')

# 標記幾個點
for q, label in [(0.9, '好'), (0.5, '普通'), (0.1, '差')]:
    ce = cross_entropy([1, 0, 0], [q, (1-q)/2, (1-q)/2])
    plt.plot(q, ce, 'ko', markersize=7)
    plt.annotate(f'q₁={q}\nCE={ce:.2f}', xy=(q, ce), xytext=(q-0.15, ce+0.5),
                 fontsize=9, arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.show()

print('AI 預測「小潔」的機率越高，誤差越小！這就是 AI 學習的方向。')

## 3-3. Cross Entropy vs 平方差（MSE）

另一種常見的誤差是**平方差（均方誤差，MSE）**：

$$\text{MSE} = \sum_{i=1}^{n} (p_i - q_i)^2$$

那 cross entropy 和 MSE 有什麼差別呢？讓我們來比較看看！

In [ ]:
def mse(P, Q):
    """均方誤差"""
    P = np.array(P, dtype=float)
    Q = np.array(Q, dtype=float)
    return np.sum((P - Q) ** 2)

# 正確答案是小潔 [1, 0, 0]
P_correct = [1, 0, 0]
q1_vals = np.linspace(0.001, 0.999, 500)

ce_vals = [cross_entropy(P_correct, [q1, (1-q1)/2, (1-q1)/2]) for q1 in q1_vals]
mse_vals = [mse(P_correct, [q1, (1-q1)/2, (1-q1)/2]) for q1 in q1_vals]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(q1_vals, ce_vals, color='crimson', linewidth=2.5)
ax1.set_xlabel('AI 預測「小潔」的機率 q₁')
ax1.set_ylabel('誤差大小')
ax1.set_title('Cross Entropy 誤差')

ax2.plot(q1_vals, mse_vals, color='steelblue', linewidth=2.5)
ax2.set_xlabel('AI 預測「小潔」的機率 q₁')
ax2.set_ylabel('誤差大小')
ax2.set_title('MSE（平方差）誤差')

plt.suptitle('當 AI 完全預測錯誤（q₁ → 0），兩種誤差的差異', fontsize=12)
plt.tight_layout()
plt.show()

print('觀察：')
print('  Cross Entropy：當 AI 完全錯誤（q₁→0），誤差 → 無限大！懲罰非常重！')
print('  MSE：當 AI 完全錯誤，誤差最多只有約 1.5，懲罰比較溫和。')
print()
print('→ 分類問題通常用 Cross Entropy，因為它對「完全自信但完全錯誤」懲罰更重！')

# 4. KL 散度：兩個分布差多遠？ 📏

## 4-1. KL 散度的定義

在機器學習中，我們常常想知道：**AI 的預測分布 $Q$ 和正確答案分布 $P$ 差多遠？**

**KL 散度（Kullback-Leibler Divergence）** 就是用來測量這個「距離」的：

$$D_{KL}(P \| Q) = H(P, Q) - H(P)$$

也可以寫成：

$$D_{KL}(P \| Q) = \sum_{i=1}^{n} p_i \log_2\left(\frac{p_i}{q_i}\right)$$

**重要性質：**
- $D_{KL}(P \| Q) \geq 0$，永遠非負
- $D_{KL}(P \| Q) = 0$ 當且僅當 $P = Q$（兩個分布完全一樣）
- **不對稱！** $D_{KL}(P \| Q) \neq D_{KL}(Q \| P)$（所以不是真正的「距離」）

In [ ]:
def kl_divergence(P, Q):
    """計算 KL 散度 D_KL(P || Q)
    注意：KL 散度不對稱，D_KL(P||Q) ≠ D_KL(Q||P)
    """
    P = np.array(P, dtype=float)
    Q = np.array(Q, dtype=float)
    mask = P > 0
    return np.sum(P[mask] * np.log2(P[mask] / Q[mask]))

# 三個分布互相比較
P = [0.5, 0.3, 0.2]   # 正確分布
Q = [0.4, 0.35, 0.25]  # 預測分布（比較接近）
R = [0.1, 0.1, 0.8]    # 另一個預測（差很多）

print('三個分布：')
print(f'  P（正確）= {P}')
print(f'  Q（接近）= {Q}')
print(f'  R（差很多）= {R}')
print()
print('熵（自身的不確定性）：')
print(f'  H(P) = {entropy(P):.4f} bits')
print(f'  H(Q) = {entropy(Q):.4f} bits')
print()
print('Cross Entropy（用 P 做標準）：')
print(f'  H(P, Q) = {cross_entropy(P, Q):.4f} bits')
print(f'  H(P, R) = {cross_entropy(P, R):.4f} bits')
print()
print('KL 散度 D_KL(P || Q)（P 和 Q 的距離）：')
print(f'  D_KL(P || Q) = {kl_divergence(P, Q):.4f} bits')
print(f'  D_KL(P || R) = {kl_divergence(P, R):.4f} bits')
print()
print('驗證關係：D_KL(P||Q) = H(P,Q) - H(P)')
print(f'  H(P,Q) - H(P) = {cross_entropy(P,Q) - entropy(P):.4f}')
print(f'  D_KL(P||Q)    = {kl_divergence(P,Q):.4f}')
print('✓ 完全一致！')

In [ ]:
# 展示 KL 散度不對稱
P = [0.9, 0.05, 0.05]
Q = [0.33, 0.33, 0.34]

d_pq = kl_divergence(P, Q)
d_qp = kl_divergence(Q, P)

print('KL 散度的不對稱性：')
print(f'  P = {P}（幾乎都是第一類）')
print(f'  Q = {Q}（均勻分布）')
print()
print(f'  D_KL(P || Q) = {d_pq:.4f}  （用 P 為基準，測 Q 和 P 差多遠）')
print(f'  D_KL(Q || P) = {d_qp:.4f}  （用 Q 為基準，測 P 和 Q 差多遠）')
print()
print(f'  → 差了 {abs(d_pq - d_qp):.4f}！KL 散度真的不對稱。')
print()
print('這也是為什麼 KL 散度不叫「距離」，而叫「散度」！')

## 4-2. 三者的關係整理

$$\underbrace{H(P, Q)}_{\text{Cross Entropy}} = \underbrace{H(P)}_{\text{Entropy}} + \underbrace{D_{KL}(P \| Q)}_{\text{KL 散度}}$$

| 名稱 | 含義 | 最小值 | 何時最小 |
|------|------|--------|----------|
| $H(P)$ 熵 | P 本身的不確定性（固定） | ≥ 0 | P 完全確定時 |
| $H(P, Q)$ Cross Entropy | 用 P 為標準，Q 的「編碼效率」 | $= H(P)$ | 當 $Q = P$ |
| $D_{KL}(P\|Q)$ KL 散度 | P 和 Q 的差異 | $= 0$ | 當 $Q = P$ |

> **AI 訓練的本質**：P 是真實資料分布，Q 是模型的預測分布。最小化 Cross Entropy 等價於最小化 KL 散度（因為 H(P) 是固定的）。

In [ ]:
# 視覺化三者的關係
P_true = [0.5, 0.3, 0.2]

# 讓 Q 的第一個元素從 0.1 到 0.9 變化（另外兩個均分）
q1_range = np.linspace(0.05, 0.90, 100)
ce_list, kl_list, h_p_list = [], [], []

h_p = entropy(P_true)

for q1 in q1_range:
    remaining = 1 - q1
    Q = [q1, remaining * 0.6, remaining * 0.4]  # 保持機率和為 1
    ce_list.append(cross_entropy(P_true, Q))
    kl_list.append(kl_divergence(P_true, Q))
    h_p_list.append(h_p)

plt.figure(figsize=(10, 5))
plt.plot(q1_range, ce_list, label='H(P,Q) Cross Entropy', color='crimson', linewidth=2)
plt.plot(q1_range, kl_list, label='D_KL(P||Q) KL 散度', color='steelblue', linewidth=2)
plt.axhline(h_p, color='green', linestyle='--', linewidth=1.5, label=f'H(P) 熵 = {h_p:.3f}（固定）')

# 標記 P 的第一個元素 q1=0.5（和 P 相同時）
plt.axvline(x=P_true[0], color='gray', linestyle=':', alpha=0.7, label=f'q₁ = p₁ = {P_true[0]}（Q≈P）')

plt.xlabel('Q 的第一個機率 q₁')
plt.ylabel('bits')
plt.title('Cross Entropy = Entropy + KL 散度')
plt.legend()
plt.tight_layout()
plt.show()

# 🎯 AI 協作挑戰

## 挑戰一：模擬三類別資料，計算熵

**題目：** 假設有一個分類問題，資料有三種類別（類別 A、B、C）。

請用 NumPy 生成以下三種資料集，並分別計算熵：

1. 均衡資料：各 100 筆（熵最大）
2. 不均衡資料：A=200筆, B=80筆, C=20筆
3. 極端不均衡：A=280筆, B=15筆, C=5筆

💡 不確定的地方問 AI：「怎麼從一組計數（counts）算出機率分布，然後計算熵？」

In [ ]:
# TODO：試試看！
datasets = {
    '均衡資料': [100, 100, 100],
    '不均衡資料': [200, 80, 20],
    '極端不均衡': [280, 15, 5],
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ['#4ECDC4', '#FFE66D', '#FF6B6B']

for ax, (name, counts) in zip(axes, datasets.items()):
    counts = np.array(counts)
    probs = counts / counts.sum()  # 轉成機率
    h = entropy(probs)

    bars = ax.bar(['類別 A', '類別 B', '類別 C'], counts, color=colors, edgecolor='white')
    ax.set_title(f'{name}\n熵 H = {h:.3f} bits', fontsize=11)
    ax.set_ylabel('資料筆數')
    for bar, n in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                str(n), ha='center', fontsize=10)

plt.suptitle('三種資料集的熵比較', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('越均衡的資料，熵越大（越難預測）；越不均衡，熵越小（越容易猜到是 A）。')

## 挑戰二：Cross Entropy vs MSE 的差異

**題目：** 正確答案是 $P = [1, 0, 0, 0]$（四類別，只有第一類是對的）。

AI 的預測是 $Q = [q_1, q_2, q_3, q_4]$，其中 $q_1 + q_2 + q_3 + q_4 = 1$。

請畫出當 $q_1$ 從 0.001 到 0.999 變化時，**cross entropy** 和 **MSE** 的比較圖。

問 AI：「為什麼分類問題通常用 cross entropy 而不是 MSE 作為損失函數？」，然後在程式跑出來之後，看看是不是符合 AI 說的原因！

In [ ]:
# TODO：完成這個比較！
P = [1, 0, 0, 0]  # 正確答案：第一類
q1_vals = np.linspace(0.001, 0.999, 500)

ce_vals = []
mse_vals = []

for q1 in q1_vals:
    remaining = (1 - q1) / 3
    Q = [q1, remaining, remaining, remaining]
    ce_vals.append(cross_entropy(P, Q))
    mse_vals.append(mse(P, Q))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(q1_vals, ce_vals, color='crimson', linewidth=2.5)
axes[0].set_xlabel('AI 預測正確類別的機率 q₁')
axes[0].set_ylabel('誤差')
axes[0].set_title('Cross Entropy 損失')
axes[0].set_xlim(0, 1)

axes[1].plot(q1_vals, mse_vals, color='steelblue', linewidth=2.5)
axes[1].set_xlabel('AI 預測正確類別的機率 q₁')
axes[1].set_ylabel('誤差')
axes[1].set_title('MSE（均方誤差）損失')
axes[1].set_xlim(0, 1)

plt.suptitle('Cross Entropy vs MSE：當 AI 完全自信但完全錯誤（q₁→0）', fontsize=12)
plt.tight_layout()
plt.show()

print('你發現了什麼？Cross Entropy 在 q₁→0 時急劇上升，MSE 則有上界。')
print('這對 AI 訓練有什麼影響？試著問 AI！')

## 挑戰三：KL 散度探索

**題目：** 生成兩個三類別的隨機機率分布 $P$ 和 $Q$，計算：

1. $H(P)$（P 的熵）
2. $H(P, Q)$（P 和 Q 的 cross entropy）
3. $D_{KL}(P \| Q)$（KL 散度）
4. 驗證：$H(P, Q) - H(P) = D_{KL}(P \| Q)$
5. 驗證：$D_{KL}(P \| Q) \neq D_{KL}(Q \| P)$

💡 可以問 AI：「用 NumPy 怎麼生成一個隨機的合法機率分布？」

In [ ]:
# TODO：自己選兩個分布試試看！
# 提示：用 np.random.dirichlet([1,1,1]) 可以生成隨機機率分布
np.random.seed(42)

P = np.random.dirichlet([1, 1, 1])  # 隨機生成合法機率分布
Q = np.random.dirichlet([1, 1, 1])

print(f'P = [{P[0]:.4f}, {P[1]:.4f}, {P[2]:.4f}]')
print(f'Q = [{Q[0]:.4f}, {Q[1]:.4f}, {Q[2]:.4f}]')
print()

h_p   = entropy(P)
h_pq  = cross_entropy(P, Q)
kl_pq = kl_divergence(P, Q)
kl_qp = kl_divergence(Q, P)

print(f'H(P)        = {h_p:.4f} bits')
print(f'H(P, Q)     = {h_pq:.4f} bits')
print(f'D_KL(P||Q)  = {kl_pq:.4f} bits')
print(f'D_KL(Q||P)  = {kl_qp:.4f} bits')
print()
print(f'驗證：H(P,Q) - H(P) = {h_pq - h_p:.4f} ≈ D_KL(P||Q) = {kl_pq:.4f}')
print(f'KL 不對稱：D_KL(P||Q) = {kl_pq:.4f} ≠ D_KL(Q||P) = {kl_qp:.4f}')

# 🧠 核心觀念回顧

| 概念 | 公式 | 直覺意義 | AI 用途 |
|------|------|---------|--------|
| **資訊量** | $I(P) = -\log_2 P$ | 機率越小 → 越驚喜 → 資訊量越大 | 霍夫曼編碼 |
| **熵** | $H(P) = -\sum p_i \log_2 p_i$ | 分布越均勻 → 越不確定 → 熵越大 | 決策樹分裂 |
| **Cross Entropy** | $H(P,Q) = -\sum p_i \log_2 q_i$ | P 和 Q 差越多 → 值越大 → 誤差越大 | 分類模型損失函數 |
| **KL 散度** | $D_{KL}(P\|Q) = H(P,Q) - H(P)$ | 兩個分布差多遠（不對稱） | 生成式 AI、VAE |

### 三者的美麗關係

$$\text{Cross Entropy} = \text{Entropy} + \text{KL 散度}$$

$$H(P, Q) = H(P) + D_{KL}(P \| Q)$$

> 最小化 Cross Entropy，等價於最小化 KL 散度（因為 $H(P)$ 是固定的）。
> 這就是為什麼 **AI 訓練 = 最小化 Cross Entropy**！

# 🌟 本週創作任務

選一個**你真的有興趣的問題**，用熵、cross entropy 或 KL 散度來探索它！

幾個有趣的方向：

- 📖 **語言的熵**：英文字母的出現頻率差很多，熵是多少？和中文相比呢？
- 🎮 **遊戲的公平性**：大富翁擲兩顆骰子的點數分布，熵是多少？一顆骰子呢？
- 🌡️ **天氣預報**：台灣各月份晴雨比例的熵，哪個月最「難預測」？
- 🃏 **撲克牌**：一副牌的某張牌是什麼花色，熵是多少？
- 🤖 **自己做個分類器**：用 `sklearn` 做一個簡單分類器，看 cross entropy loss 如何下降

請在報告中回答：

1️⃣ 你想探索的**問題**是什麼？

2️⃣ 你用了**哪個概念**（熵？cross entropy？KL 散度？）？為什麼？

3️⃣ 你**發現**了什麼有趣的事？有沒有出乎意料？

4️⃣ 這週你**問了 AI 什麼**？AI 的回答有沒有讓你更理解這些概念？

祝你玩得開心，並感受到熵的魔力！✨